In [ ]:
# Grouped and deduplicated for clarity. Feel free to tweak.
import os, multiprocessing as mp
cores = mp.cpu_count()
os.environ["TF_NUM_INTRAOP_THREADS"] = str(cores)  # parallel within ops
os.environ["TF_NUM_INTEROP_THREADS"] = "2"         # parallel across ops
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "1"          # Intel-optimized kernels

import tensorflow as tf
tf.config.optimizer.set_jit(True)  # try XLA on CPU
gpus = tf.config.experimental.list_physical_devices("GPU")
print("TF:", tf.__version__)

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPUs detected: {len(gpus)}")
    except RuntimeError as e:
        print("Error setting memory growth:", e)
else:
    print("No GPU detected")

# --- Standard Library ---
import os; import sys; import logging; import time; import numpy as np; import tensorflow as tf
from datetime import datetime
import time; import numpy as np
import os; import tensorflow as tf
import os; import io; import contextlib; import tensorflow as tf
import os
import os; import numpy as np; import matplotlib.pyplot as plt
import os; import numpy as np
import math
# --- Scientific Stack ---
import numpy as np
# --- Plotting ---
import matplotlib.pyplot as plt
# --- Scikit-learn & Metrics ---
from sklearn.model_selection import train_test_split
# --- Other Third-Party ---
from keras.models import Model
from keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate, Conv2DTranspose, BatchNormalization, Dropout, Lambda
from keras import backend as K
from src.models.utils import show_random_datapoint
from models.lossfunc import jaccard_coef, dice_coef, dice_loss, jaccard_coef_loss, FocalLoss, bce_loss, bce_dice, bce_jaccard, bce_focal, bce_dice_focal
from models.unet import simple_unet_model
from models.utils import read_images_and_masks, show_random_datapoint
import tensorflow as tf; import gc
from tensorflow.keras.callbacks import Callback
from tensorflow.keras.callbacks import CSVLogger
from tensorflow.keras.callbacks import Callback, EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.metrics import BinaryAccuracy, Precision, Recall

In [ ]:
image_directory = r"/home/anvy4548/projects/crystal-recognition/patches_for_training/images/"  # Change this to your output images directory
mask_directory = r"/home/anvy4548/projects/crystal-recognition/patches_for_training/masks/"  # Change this to your output masks directory

SIZE = 256

image_dataset, mask_dataset = read_images_and_masks(image_directory, mask_directory, SIZE)

print("Image dataset size: " + str(image_dataset.shape))
print("Mask dataset size: " + str(mask_dataset.shape))

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(image_dataset, mask_dataset, test_size = 0.10, random_state = 0)
X_train = X_train.astype('float32'); X_test = X_test.astype('float32')
y_train = y_train.astype('float32'); y_test = y_test.astype('float32')

IMG_HEIGHT = image_dataset.shape[1]
IMG_WIDTH  = image_dataset.shape[2]
IMG_CHANNELS = image_dataset.shape[3]

In [ ]:
show_random_datapoint(image_dataset, mask_dataset)

In [ ]:
# metrics to track
metrics = [dice_coef, jaccard_coef]

model_bce         = simple_unet_model(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS,
                                      loss='binary_crossentropy', metrics=metrics)

model_bce_fn      = simple_unet_model(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS,
                                      loss=bce_loss, metrics=metrics)  # same as above but via callable

model_dice        = simple_unet_model(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS,
                                      loss=dice_loss, metrics=metrics)

model_jaccard     = simple_unet_model(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS,
                                      loss=jaccard_coef_loss, metrics=metrics)

model_focal       = simple_unet_model(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS,
                                      loss=FocalLoss, metrics=metrics)


model_bce_dice    = simple_unet_model(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS,
                                      loss=bce_dice, metrics=metrics)

model_bce_jaccard = simple_unet_model(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS,
                                      loss=bce_jaccard, metrics=metrics)

model_bce_focal   = simple_unet_model(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS,
                                      loss=bce_focal, metrics=metrics)

model_bce_dice_focal = simple_unet_model(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS,
                                         loss=bce_dice_focal, metrics=metrics)


In [ ]:
from pathlib import Path

# --- directories ---
outdir = Path("outputs")
logdir = outdir / "logs"
ckptdir = outdir / "checkpoints"
logdir.mkdir(parents=True, exist_ok=True)
ckptdir.mkdir(parents=True, exist_ok=True)

# --- log file path ---
run_id = datetime.now().strftime("%Y%m%d-%H%M%S")
log_path = logdir / f"train_{run_id}.log"

# --- logger setup ---
logger = logging.getLogger("train")
logger.setLevel(logging.INFO)
fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")  # Stockholm local time
fh = logging.FileHandler(log_path); fh.setFormatter(fmt)
sh = logging.StreamHandler(sys.stdout); sh.setFormatter(fmt)
logger.handlers.clear(); logger.addHandler(fh); logger.addHandler(sh)

# --- helper for pretty printing metrics ---
def fmt_logs(logs):
    logs = logs or {}
    parts = [f"{k}={v:.4f}" if isinstance(v, (float, np.floating)) else f"{k}={v}"
             for k, v in sorted(logs.items())]
    return " ".join(parts)

# --- Keras callback for logging ---
class LogCallback(Callback):
    def on_train_begin(self, logs=None):
        logger.info("Training start | params=%s", self.params)
    def on_epoch_begin(self, epoch, logs=None):
        self._t0 = time.perf_counter()
    def on_epoch_end(self, epoch, logs=None):
        dt = time.perf_counter() - self._t0
        logger.info("epoch=%d | %s | epoch_sec=%.2f", epoch, fmt_logs(logs), dt)
    def on_train_end(self, logs=None):
        logger.info("Training end")

In [ ]:
import os, json
from pathlib import Path
from datetime import datetime
from tensorflow.keras.callbacks import CSVLogger, Callback

# --- CSV logger with timestamp column ---
class CSVLoggerWithTime(CSVLogger):
    def on_epoch_end(self, epoch, logs=None):
        logs = dict(logs or {})
        logs["timestamp"] = datetime.now().isoformat(timespec="seconds")
        super().on_epoch_end(epoch, logs)

# --- JSON summary saver (runs once after training) ---
class JSONSummary(Callback):
    def __init__(self, outpath, config=None, timer=None):
        super().__init__()
        self.outpath = Path(outpath)
        self.config = config or {}
        self.timer = timer   # optional EpochTimer callback

    def on_train_end(self, logs=None):
        summary = {
            "finished_at": datetime.now().isoformat(timespec="seconds"),
            "params": self.params,
            "config": self.config,
        }
        if self.timer is not None:
            summary["runtime"] = {
                "epochs_ran": len(self.timer.epoch_times),
                "total_sec": float(sum(self.timer.epoch_times)),
                "mean_epoch_sec": float(np.mean(self.timer.epoch_times)),
            }
        self.outpath.parent.mkdir(parents=True, exist_ok=True)
        with self.outpath.open("w") as f:
            json.dump(summary, f, indent=2)

# --- where to save ---
logdir = Path("outputs/logs"); logdir.mkdir(parents=True, exist_ok=True)
histdir = Path("outputs/histories"); histdir.mkdir(parents=True, exist_ok=True)

# unique ID for this run
run_id = datetime.now().strftime("%Y%m%d-%H%M%S")

# callbacks to pass into model.fit()
csv_cb = CSVLoggerWithTime(str(logdir / f"history_{run_id}.csv"))
json_cb = JSONSummary(histdir / f"summary_{run_id}.json", config={
    "img_h": IMG_HEIGHT, "img_w": IMG_WIDTH, "channels": IMG_CHANNELS,
    "batch_size": 16, "epochs": 10,
})


In [ ]:
print("X_train", X_train.shape, X_train.dtype, X_train.min(), X_train.max())
print("y_train", y_train.shape, y_train.dtype, y_train.min(), y_train.max())
print("pos frac train:", float(y_train.mean()), " | pos frac val:", float(y_test.mean()))


In [ ]:
import os, sys, gc, time, json
import numpy as np
from pathlib import Path
from datetime import datetime
import tensorflow as tf
from tensorflow.keras.callbacks import (
    Callback, EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, CSVLogger
)
from tensorflow.keras.metrics import BinaryAccuracy, Precision, Recall

# ============================================================
# --- Custom callbacks ---
# ============================================================

class EpochTimer(Callback):
    """Measure duration of each epoch."""
    def on_train_begin(self, logs=None):
        self.epoch_times = []
    def on_epoch_begin(self, epoch, logs=None):
        self._t0 = time.perf_counter()
    def on_epoch_end(self, epoch, logs=None):
        self.epoch_times.append(time.perf_counter() - self._t0)

class SegDebug(Callback):
    """Quick debug of segmentation quality on a small validation subset."""
    def __init__(self, x_val, y_val, n=64, thr=0.5):
        super().__init__()
        self.x = x_val[:n]; self.y = y_val[:n]; self.thr = thr
    def on_train_begin(self, logs=None):
        print(f"[dbg] y_val pos frac: {float(self.y.mean()):.3f}")
    def on_epoch_end(self, epoch, logs=None):
        p = self.model.predict(self.x, verbose=0)
        pm = float(p.mean()); pf = float((p > self.thr).mean())
        inter = float(np.logical_and(p > self.thr, self.y > 0.5).mean())
        print(f"[dbg] epoch {epoch:03d} pred_mean={pm:.3f}  "
              f"pred>thr={pf:.3f}  IoU(quick)≈{inter:.3f}")

# CSV logger that also adds a timestamp column
class CSVLoggerWithTime(CSVLogger):
    def on_epoch_end(self, epoch, logs=None):
        logs = dict(logs or {})
        logs["timestamp"] = datetime.now().isoformat(timespec="seconds")
        super().on_epoch_end(epoch, logs)

# JSON summary saver (writes one JSON per training run)
class JSONSummary(Callback):
    def __init__(self, outpath, config=None, timer=None):
        super().__init__()
        self.outpath = Path(outpath)
        self.config = config or {}
        self.timer = timer   # optional EpochTimer instance
    def on_train_end(self, logs=None):
        summary = {
            "finished_at": datetime.now().isoformat(timespec="seconds"),
            "params": self.params,
            "config": self.config,
        }
        if self.timer is not None:
            summary["runtime"] = {
                "epochs_ran": len(self.timer.epoch_times),
                "total_sec": float(sum(self.timer.epoch_times)),
                "mean_epoch_sec": float(np.mean(self.timer.epoch_times)),
            }
        self.outpath.parent.mkdir(parents=True, exist_ok=True)
        with self.outpath.open("w") as f:
            json.dump(summary, f, indent=2)

# ============================================================
# --- Directories ---
# ============================================================

logdir = Path("outputs/logs"); logdir.mkdir(parents=True, exist_ok=True)
histdir = Path("outputs/histories"); histdir.mkdir(parents=True, exist_ok=True)
ckptdir = Path("outputs/checkpoints"); ckptdir.mkdir(parents=True, exist_ok=True)

run_id = datetime.now().strftime("%Y%m%d-%H%M%S")

# ============================================================
# --- Metrics and losses ---
# ============================================================

metrics = [
    dice_coef,
    jaccard_coef,
    BinaryAccuracy(name='accuracy', threshold=0.5),
    Precision(name='prec'),
    Recall(name='rec')
]

losses = {
    "bce_dice_focal": bce_dice_focal,
    "dice": dice_loss,
#    "jaccard": jaccard_coef_loss,
    "focal": FocalLoss,
    "bce": "binary_crossentropy",
    "bce_dice": bce_dice,
    "bce_jaccard": bce_jaccard,
    "bce_focal": bce_focal
}

models = {
    n: simple_unet_model(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS,
                         loss=f, metrics=metrics)
    for n, f in losses.items()
}

# ============================================================
# --- Training loop ---
# ============================================================

histories, runtimes = {}, {}

for name, model in models.items():
    tf.keras.backend.clear_session()
    print(f"\n=== {name} ===")

    # timer + loggers
    timer = EpochTimer()
    csv_cb = CSVLoggerWithTime(str(logdir / f"history_{name}_{run_id}.csv"))
    json_cb = JSONSummary(
        histdir / f"summary_{name}_{run_id}.json",
        config={
            "loss": name,
            "img_h": IMG_HEIGHT, "img_w": IMG_WIDTH,
            "channels": IMG_CHANNELS,
            "batch_size": 16, "epochs": 10,
        },
        timer=timer
    )

    # callbacks
    cbs = [
        EarlyStopping(patience=20, restore_best_weights=True, monitor="val_loss"),
        ReduceLROnPlateau(patience=5, factor=0.5, min_lr=1e-6, monitor="val_loss"),
        ModelCheckpoint(str(ckptdir / f"unet_{name}"),
                        monitor="val_loss", save_best_only=True,
                        save_weights_only=False),
        timer,
        SegDebug(X_test, y_test, n=64, thr=0.5),
        LogCallback(),  # your structured console/file logger
        csv_cb,
        json_cb,
    ]

    # fit
    t0 = time.perf_counter()
    hist = model.fit(
        X_train, y_train,
        batch_size=16,
        epochs=30,
        validation_data=(X_test, y_test),
        shuffle=True,
        verbose=1,
        callbacks=cbs,
    )
    total = time.perf_counter() - t0

    # store minimal history in RAM (optional, since CSV covers it)
    histories[name] = {
        "loss": hist.history["loss"],
        "val_loss": hist.history["val_loss"],
    }
    runtimes[name] = {
        "total_sec": total,
        "epochs_ran": len(timer.epoch_times),
        "mean_epoch_sec": float(np.mean(timer.epoch_times)),
        "last_epoch_sec": float(timer.epoch_times[-1]),
    }

    # cleanup
    del model, hist
    tf.keras.backend.clear_session(); gc.collect()

# ============================================================
# --- Quick summary ---
# ============================================================

for k, v in runtimes.items():
    print(f"{k:16s}  total: {v['total_sec']:.1f}s | "
          f"epochs: {v['epochs_ran']} | mean/epoch: {v['mean_epoch_sec']:.2f}s")


In [ ]:
from pathlib import Path
import csv, re, math
from statistics import mean

logdir = Path("outputs/logs")
hist_files = sorted(logdir.glob("history_linknet*.csv"))

def to_float(x):
    try:
        return float(x)
    except Exception:
        return math.nan

def read_history_plain(path: Path):
    rows = []
    with path.open("r", encoding="utf-8", errors="replace") as f:
        rdr = csv.DictReader(f)
        for r in rdr:
            # normalize types
            r2 = {k: r[k] for k in r}
            # ints
            if "epoch" in r2 and r2["epoch"] != "":
                try: r2["epoch"] = int(float(r2["epoch"]))
                except: r2["epoch"] = None
            # floats for common metrics
            for k in ["loss","val_loss","dice_coef","val_dice_coef","jaccard_coef","val_jaccard_coef",
                      "acc","val_acc","prec","val_prec","rec","val_rec","lr"]:
                if k in r2:
                    r2[k] = to_float(r2[k])
            rows.append(r2)
    return rows

summaries = []
failures = []

for f in hist_files:
    try:
        hist = read_history_plain(f)
        if not hist:
            raise ValueError("empty after parse")
        # find best epoch by lowest val_loss (skip NaNs)
        best_idx, best_val = None, float("inf")
        for i, r in enumerate(hist):
            v = r.get("val_loss", float("inf"))
            if v == v and v < best_val:  # v==v filters NaN
                best_val, best_idx = v, i
        if best_idx is None:
            raise ValueError("no valid val_loss values")

        best = hist[best_idx]
        # loss name from filename
        m = re.search(r"history_(.+?)_\d{8}-\d{6}\.csv$", f.name)
        loss_name = m.group(1) if m else "unknown"

        # timings if present (you added timestamp per-epoch)
        timestamps = [r.get("timestamp") for r in hist if r.get("timestamp")]
        summaries.append({
            "loss": loss_name,
            "csv": str(f),
            "epochs": len(hist),
            "best_epoch": int(best.get("epoch", best_idx)),
            "best_val_loss": float(best.get("val_loss")),
            "train_loss_at_best": float(best.get("loss", float("nan"))),
            "val_dice_at_best": float(best.get("val_dice_coef", float("nan"))),
            "val_jaccard_at_best": float(best.get("val_jaccard_coef", float("nan"))),
            "val_acc_at_best": float(best.get("val_acc", float("nan"))),
            "val_prec_at_best": float(best.get("val_prec", float("nan"))),
            "val_rec_at_best": float(best.get("val_rec", float("nan"))),
            "first_timestamp": timestamps[0] if timestamps else None,
            "last_timestamp": timestamps[-1] if timestamps else None,
        })
    except Exception as e:
        failures.append((f.name, str(e)))

# print table
if summaries:
    summaries.sort(key=lambda d: d["best_val_loss"])
    cols = ["loss","epochs","best_epoch","best_val_loss","val_dice_at_best","val_jaccard_at_best","val_acc_at_best"]
    header = " | ".join(f"{c:>18s}" for c in cols)
    print(header)
    print("-"*len(header))
    for s in summaries:
        print(" | ".join([
            f"{s['loss'][:18]:>18s}",
            f"{s['epochs']:>18d}",
            f"{s['best_epoch']:>18d}",
            f"{s['best_val_loss']:>18.6f}",
            f"{(s['val_dice_at_best'] if s['val_dice_at_best']==s['val_dice_at_best'] else float('nan')):>18.6f}",
            f"{(s['val_jaccard_at_best'] if s['val_jaccard_at_best']==s['val_jaccard_at_best'] else float('nan')):>18.6f}",
            f"{(s['val_acc_at_best'] if s['val_acc_at_best']==s['val_acc_at_best'] else float('nan')):>18.6f}",
        ]))
else:
    print("No histories parsed.")

if failures:
    print("\nSkipped files:")
    for name, err in failures:
        print(f"- {name}: {err}")


In [ ]:
import os; os.environ['SM_FRAMEWORK']='tf.keras'
import segmentation_models as sm
print(sm.framework())  # should print 'tf.keras'


In [ ]:
# Super-minimal Keras training loop (DRY, no frills) — UNet++
# Assumes these exist in your environment:
#   simple_unetpp_model (preferred), X_train, y_train, X_test, y_test,
#   dice_coef, jaccard_coef, dice_loss, jaccard_coef_loss,
#   FocalLoss, bce_dice_focal, bce_dice, bce_jaccard, bce_focal

import time
from pathlib import Path
from datetime import datetime
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, CSVLogger
from tensorflow.keras.metrics import BinaryAccuracy, Precision, Recall
from keras_unet_collection import models
import tensorflow as tf
import os
os.environ["SM_FRAMEWORK"] = "tf.keras"   # must be set before import

import segmentation_models as sm
sm.set_framework('tf.keras')

# Optional: fallback UNet++ builder using `segmentation_models`
def linknet_sm(h, w, c, loss, metrics):
    model = sm.Linknet(
        backbone_name='resnet34',
        input_shape=(h, w, c),
        classes=1,
        activation='sigmoid',
        encoder_weights=None,  # for 1-channel inputs
    )
    model.compile(optimizer=tf.keras.optimizers.Adam(), loss=loss, metrics=metrics)
    return model

def make_model(loss_fn):
    return linknet_sm(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS, loss=loss_fn, metrics=metrics)



# ====== basics ======
IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS = IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS  # reuse your globals
BATCH, EPOCHS = 16, 300
RUN_ID = datetime.now().strftime("%Y%m%d-%H%M%S")
BASE = Path("outputs"); LOGS = BASE/"logs"; CKPTS = BASE/"checkpoints"
for p in (LOGS, CKPTS): p.mkdir(parents=True, exist_ok=True)

# ====== metrics/losses (plug yours) ======
metrics = [dice_coef, jaccard_coef, BinaryAccuracy(name="acc", threshold=0.5), Precision(name="prec"), Recall(name="rec")]
losses = {
    #"bce_dice_focal": bce_dice_focal,
    "dice": dice_loss,
    "jaccard": jaccard_coef_loss,
    "focal": FocalLoss,
    "bce": "binary_crossentropy",
    "bce_dice": bce_dice,
    "bce_jaccard": bce_jaccard,
    "bce_focal": bce_focal,
}
/
# ====== small helpers ======


ES_PATIENCE = 12
RLR_PATIENCE = 6

def make_callbacks(name, es_patience=ES_PATIENCE, rlr_patience=RLR_PATIENCE):
    return [
        EarlyStopping(monitor="val_loss", patience=es_patience, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", patience=rlr_patience, factor=0.5, min_lr=1e-6),
        ModelCheckpoint(str(CKPTS / f"linknet_{name}"), monitor="val_loss", save_best_only=True),
        CSVLogger(str(LOGS / f"history_linknet_{name}_{RUN_ID}.csv")),
    ]

# ====== train all losses ======
for name, loss_fn in losses.items():
    tf.keras.backend.clear_session()
    print(f"\n=== {name} ===")
    model = make_model(loss_fn)
    t0 = time.perf_counter()
    hist = model.fit(
        X_train, y_train,
        batch_size=BATCH,
        epochs=EPOCHS,
        validation_data=(X_test, y_test),
        shuffle=True,
        callbacks=make_callbacks(name),
        verbose=1,
    )
    dt = time.perf_counter() - t0
    print(f"{name:16s} total: {dt:.1f}s | epochs: {len(hist.history['loss'])}")

# Done. Keep it lean; read CSVs in notebooks for plots/analysis.


In [ ]:
import matplotlib.pyplot as plt

for s in summaries:
    hist = read_history_plain(Path(s["csv"]))
    epochs = [r.get("epoch", i) if r.get("epoch") is not None else i for i,r in enumerate(hist)]

    # losses
    tr_loss = [r.get("loss", float("nan")) for r in hist]
    va_loss = [r.get("val_loss", float("nan")) for r in hist]

    # accuracies
    tr_acc = [r.get("acc", float("nan")) for r in hist]
    va_acc = [r.get("val_acc", float("nan")) for r in hist]

    plt.figure(figsize=(10,4))

    # left subplot: loss
    plt.subplot(1,2,1)
    plt.plot(epochs, tr_loss, label="train_loss")
    plt.plot(epochs, va_loss, label="val_loss")
    plt.title(f"{s['loss']} – Loss")
    plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()

    # right subplot: accuracy
    plt.subplot(1,2,2)
    plt.plot(epochs, tr_acc, label="train_acc")
    plt.plot(epochs, va_acc, label="val_acc")
    plt.title(f"{s['loss']} – Accuracy")
    plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.legend()

    plt.tight_layout()
    plt.show()


In [ ]:
import time, numpy as np

def _soft_iou_from_probs(probs, y_true):
    y = (y_true > 0.5).astype(np.float32)
    num = np.sum(probs * y, axis=(1,2,3))
    den = np.sum(probs + y - probs * y, axis=(1,2,3))
    return float(np.mean((num + 1e-7) / (den + 1e-7)))

def _hard_iou_from_probs(probs, y_true, thr):
    y = (y_true > 0.5)
    p = (probs >= thr)
    inter = np.sum(y & p, axis=(1,2,3))
    union = np.sum(y | p, axis=(1,2,3))
    valid = union > 0
    iou = np.where(valid, inter / (union + 1e-7), 1.0)   # define IoU=1 if both empty
    return float(np.mean(iou))

def evaluate_all(models, X, y, thresholds=np.round(np.linspace(0.3, 0.7, 9), 2)):
    results = []
    for name, m in models.items():
        # Keras metrics (whatever you compiled with)
        eval_res = m.evaluate(X, y, verbose=0, return_dict=True)

        # Predictions + timing
        t0 = time.perf_counter()
        probs = m.predict(X, verbose=0)
        pred_time = time.perf_counter() - t0

        # IoUs
        soft = _soft_iou_from_probs(probs, y)
        hard_scores = {t: _hard_iou_from_probs(probs, y, t) for t in thresholds}
        best_t = max(hard_scores, key=hard_scores.get)

        results.append({
            "model": name,
            **{k: float(v) for k, v in eval_res.items()},
            "soft_iou": soft,
            "best_hard_iou": hard_scores[best_t],
            "best_thr": float(best_t),
            "pred_sec_total": pred_time,
            "pred_sec_per_image": pred_time / len(X),
        })

    # sort by best hard IoU, desc
    return sorted(results, key=lambda d: d["best_hard_iou"], reverse=True)

# run it
summary = evaluate_all(models, X_test, y_test)

# print a quick leaderboard
for r in summary:
    print(f"{r['model']:16s} loss={r['loss']:.4f} softIoU={r['soft_iou']:.4f} "
          f"bestIoU@{r['best_thr']:.2f}={r['best_hard_iou']:.4f} "
          f"pred/img={r['pred_sec_per_image']*1000:.1f} ms")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

def show_random_preds(models, X, y, thr=0.6, idx=None, show_probs=False,
                      sort_by="iou", max_cols=6, seed=None, savepath=None):
    """
    models   : dict[name -> keras.Model]
    X, y     : arrays; y may be (H,W) or (H,W,1). Values in {0,1} or [0,1]
    thr      : threshold for hard masks
    idx      : choose index; None picks random
    show_probs : overlay probability map instead of hard mask
    sort_by  : "iou" | "name" | None
    max_cols : max subplot columns
    seed     : RNG seed for reproducible idx when idx=None
    savepath : if set, saves figure instead of plt.show()
    """
    rng = np.random.default_rng(seed)
    if idx is None:
        idx = int(rng.integers(0, len(X)))

    # --- get img/gt with robust squeezing ---
    img = np.squeeze(X[idx])       # (H,W) or (H,W,C)
    gt  = np.squeeze(y[idx]) > 0.5 # boolean (H,W)

    # ensure network input has shape (1,H,W,1 or C)
    if img.ndim == 2:
        img_in = img[np.newaxis, ..., np.newaxis]
        img_disp = img
    elif img.ndim == 3:
        img_in = img[np.newaxis, ...]
        img_disp = img[..., 0]     # show first channel
    else:
        raise ValueError(f"Unexpected image shape: {img.shape}")

    preds = []
    gt_sum = int(gt.sum())

    for name, m in models.items():
        prob = np.squeeze(m.predict(img_in, verbose=0))   # (H,W) or (H,W,1)
        if prob.ndim == 3:
            prob = prob[..., 0]
        mask = prob >= thr

        inter = int(np.logical_and(gt, mask).sum())
        union = int(np.logical_or(gt, mask).sum())
        iou  = (inter / union) if union > 0 else 1.0
        denom = gt_sum + int(mask.sum())
        dice = (2.0 * inter / denom) if denom > 0 else 1.0

        preds.append((name, prob, mask, float(iou), float(dice)))

    # optional sorting
    if sort_by == "iou":
        preds.sort(key=lambda t: t[3], reverse=True)
    elif sort_by == "name":
        preds.sort(key=lambda t: t[0])

    # --- layout ---
    panels = 2 + len(preds)
    cols = min(panels, max_cols)
    rows = math.ceil(panels / cols)
    fig = plt.figure(figsize=(4 * cols, 4 * rows))

    vmin, vmax = float(img_disp.min()), float(img_disp.max())

    # 1) images image
    ax = plt.subplot(rows, cols, 1)
    ax.set_title(f"Image #{idx}")
    ax.imshow(img_disp, cmap="gray", vmin=vmin, vmax=vmax); ax.axis("off")

    # 2) ground truth
    ax = plt.subplot(rows, cols, 2)
    ax.set_title("Ground truth")
    ax.imshow(img_disp, cmap="gray", vmin=vmin, vmax=vmax)
    ax.imshow(gt, alpha=0.4); ax.axis("off")

    # 3..) each model
    for k, (name, prob, mask, iou, dice) in enumerate(preds, start=3):
        ax = plt.subplot(rows, cols, k)
        ax.set_title(f"{name}\nIoU@{thr:.2f}={iou:.3f}  Dice={dice:.3f}")
        ax.imshow(img_disp, cmap="gray", vmin=vmin, vmax=vmax)
        overlay = prob if show_probs else mask
        im = ax.imshow(overlay, alpha=0.4)
        ax.axis("off")
        if show_probs:
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    if savepath:
        if show_probs:

            plt.savefig(f'{savepath}/unet_{idx}_heatmap.png', dpi=160, bbox_inches="tight")
            plt.show()
            plt.close(fig)
        else:
            plt.savefig(f'{savepath}/unet_{idx}', dpi=160, bbox_inches="tight")
            plt.show()
            plt.close(fig)
    else:
        plt.show()


# usage:
show_random_preds(models, X_test, y_test, thr=0.6)
show_random_preds(models, X_test, y_test, thr=0.6, show_probs=True)  # heatmaps


In [ ]:
idx = np.random.randint(len(X_test))  # one random image
output_folder = r'outputs/training_result_images'
show_random_preds(models, X_test, y_test, thr=0.6, idx=idx, sort_by="name", savepath=output_folder)
show_random_preds(models, X_test, y_test, thr=0.6, idx=idx, show_probs=True, sort_by="name", savepath=output_folder)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
from skimage.morphology import remove_small_holes, remove_small_objects, binary_closing, disk
from skimage.measure import label, regionprops
from skimage.morphology import binary_dilation

# ============================================================
# --- Postprocessing with hole filling + contrast filter ---
# ============================================================

def clean_mask_with_contrast(P, img, *,
                             thr=0.5,
                             hole_area=500,      # fill holes smaller than this
                             min_blob=300,       # restore blobs in [min_blob, max_blob]
                             max_blob=5000,
                             prob_bg=0.30,       # mean prob inside blob must be <= this
                             contrast_thr=0.1,   # relative intensity contrast
                             ring_r=3,           # width of ring outside hole
                             close_r=3,          # small closing
                             min_fg=150):        # remove tiny specks
    """
    P     : probability map (H,W)
    img   : grayscale input image (H,W)
    thr   : base threshold for foreground
    """
    m0 = P > thr
    m1 = binary_closing(m0, disk(close_r)) if close_r else m0
    m_filled = remove_small_holes(m1, area_threshold=hole_area)

    # candidate filled holes
    added = m_filled & (~m1)
    lab = label(added, connectivity=1)
    restore = np.zeros_like(m_filled, dtype=bool)

    rng = float(img.max() - img.min() + 1e-8)

    for reg in regionprops(lab):
        a = reg.area
        if not (min_blob <= a <= max_blob):
            continue

        rr, cc = zip(*reg.coords)
        hole_mask = np.zeros_like(m_filled, bool)
        hole_mask[rr, cc] = True

        # check model confidence inside
        mean_p = float(np.mean(P[rr, cc]))
        if mean_p > prob_bg:
            continue  # too grid-like, don’t restore

        # check contrast
        ring = binary_dilation(hole_mask, disk(ring_r)) & ~hole_mask
        if ring.sum() == 0:
            continue
        I_in  = img[hole_mask].mean()
        I_out = img[ring].mean()
        contrast = abs(I_in - I_out) / rng

        if contrast >= contrast_thr:
            # strong contrast → looks like a real object → restore hole
            restore[rr, cc] = True

    # final mask
    m_final = m_filled & (~restore)
    m_final = remove_small_objects(m_final, min_size=min_fg)
    return m_final


# ============================================================
# --- Viewer function with optional postprocessing ---
# ============================================================

def show_random_preds(models, X, y, thr=0.6, idx=None, show_probs=False,
                      sort_by="iou", max_cols=6, seed=None, savepath=None,
                      use_post=True, **pp_kwargs):
    """
    models   : dict[name -> keras.Model]
    X, y     : arrays; y may be (H,W) or (H,W,1). Values in {0,1} or [0,1]
    thr      : threshold for hard masks
    idx      : choose index; None picks random
    show_probs : overlay probability map instead of hard mask
    sort_by  : "iou" | "name" | None
    max_cols : max subplot columns
    seed     : RNG seed
    savepath : save directory
    use_post : whether to run postprocessing
    pp_kwargs: kwargs forwarded to clean_mask_with_contrast
    """
    rng = np.random.default_rng(seed)
    if idx is None:
        idx = int(rng.integers(0, len(X)))

    # --- get img/gt with robust squeezing ---
    img = np.squeeze(X[idx])       # (H,W) or (H,W,C)
    gt  = np.squeeze(y[idx]) > 0.5 # boolean (H,W)

    # ensure network input has shape (1,H,W,C)
    if img.ndim == 2:
        img_in = img[np.newaxis, ..., np.newaxis]
        img_disp = img
    elif img.ndim == 3:
        img_in = img[np.newaxis, ...]
        img_disp = img[..., 0]     # show first channel
    else:
        raise ValueError(f"Unexpected image shape: {img.shape}")

    preds = []
    gt_sum = int(gt.sum())

    for name, m in models.items():
        prob = np.squeeze(m.predict(img_in, verbose=0))
        if prob.ndim == 3:
            prob = prob[..., 0]

        mask = prob >= thr
        if use_post:
            mask = clean_mask_with_contrast(prob, img_disp, thr=thr, **pp_kwargs)

        inter = int(np.logical_and(gt, mask).sum())
        union = int(np.logical_or(gt, mask).sum())
        iou  = (inter / union) if union > 0 else 1.0
        denom = gt_sum + int(mask.sum())
        dice = (2.0 * inter / denom) if denom > 0 else 1.0

        preds.append((name, prob, mask, float(iou), float(dice)))

    # optional sorting
    if sort_by == "iou":
        preds.sort(key=lambda t: t[3], reverse=True)
    elif sort_by == "name":
        preds.sort(key=lambda t: t[0])

    # --- layout ---
    panels = 2 + len(preds)
    cols = min(panels, max_cols)
    rows = math.ceil(panels / cols)
    fig = plt.figure(figsize=(4 * cols, 4 * rows))

    vmin, vmax = float(img_disp.min()), float(img_disp.max())

    # 1) images image
    ax = plt.subplot(rows, cols, 1)
    ax.set_title(f"Image #{idx}")
    ax.imshow(img_disp, cmap="gray", vmin=vmin, vmax=vmax); ax.axis("off")

    # 2) ground truth
    ax = plt.subplot(rows, cols, 2)
    ax.set_title("Ground truth")
    ax.imshow(img_disp, cmap="gray", vmin=vmin, vmax=vmax)
    ax.imshow(gt, alpha=0.4); ax.axis("off")

    # 3..) each model
    for k, (name, prob, mask, iou, dice) in enumerate(preds, start=3):
        ax = plt.subplot(rows, cols, k)
        tag = " +pp" if use_post else ""
        ax.set_title(f"{name}{tag}\nIoU={iou:.3f}  Dice={dice:.3f}")
        ax.imshow(img_disp, cmap="gray", vmin=vmin, vmax=vmax)
        overlay = prob if show_probs else mask
        im = ax.imshow(overlay, alpha=0.4)
        ax.axis("off")
        if show_probs:
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    if savepath:
        suffix = "heatmap" if show_probs else "mask"
        plt.savefig(f"{savepath}/unet_{idx}_{suffix}_postproc.png", dpi=160, bbox_inches="tight")
        plt.show()
        plt.close(fig)
    else:
        plt.show()


In [ ]:
idx = np.random.randint(len(X_test))  # one random image
output_folder = r'outputs/training_result_images'
show_random_preds(models, X_test, y_test, thr=0.6, idx=idx, sort_by="name", savepath=output_folder)
show_random_preds(models, X_test, y_test, thr=0.6, idx=idx, show_probs=True, use_post=False, sort_by="name", savepath=output_folder)

In [ ]:
from pathlib import Path
from tensorflow.keras.models import load_model

# point to your checkpoint root
ckptdir = Path("outputs/checkpoints")

# (optional) if you only need predict(), skip compiling & custom_objects
def load_all_unets(ckptdir):
    models = {}
    for d in sorted(ckptdir.glob("unet_*")):
        if d.is_dir():
            name = d.name.replace("unet_", "")
            print(f"Loading {name} from {d}")
            models[name] = load_model(d, compile=False)  # SavedModel folder
    return models

models = load_all_unets(ckptdir)
print("Loaded:", list(models.keys()))


In [ ]:
import os, cv2, math, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from tensorflow.keras.utils import normalize
from skimage.morphology import remove_small_holes, remove_small_objects, binary_closing, disk, binary_dilation
from skimage.measure import label, regionprops

# ==============================
# Post-processing (optional)
# ==============================
def clean_mask_with_contrast(P, img, *,
                             thr=0.5,
                             hole_area=600,
                             min_blob=400, max_blob=6000,
                             prob_bg=0.30,
                             contrast_thr=0.10,
                             ring_r=3, close_r=3, min_fg=150):
    m0 = P > thr
    m1 = binary_closing(m0, disk(close_r)) if close_r else m0
    m_filled = remove_small_holes(m1, area_threshold=hole_area)

    added = m_filled & (~m1)
    lab = label(added, connectivity=1)
    restore = np.zeros_like(m_filled, dtype=bool)

    rng = float(img.max() - img.min() + 1e-8)
    for reg in regionprops(lab):
        a = reg.area
        if not (min_blob <= a <= max_blob):  # size gate in pixels
            continue
        rr, cc = zip(*reg.coords)
        mean_p = float(np.mean(P[rr, cc]))
        if mean_p > prob_bg:  # looks like foreground grid → don't restore
            continue
        ring = binary_dilation(np.array(reg.image, dtype=bool), disk(ring_r))
        # paste ring in full coords
        hole_mask = np.zeros_like(m_filled, bool); hole_mask[rr, cc] = True
        ring_full = binary_dilation(hole_mask, disk(ring_r)) & ~hole_mask
        if ring_full.sum() == 0: continue
        I_in, I_out = img[hole_mask].mean(), img[ring_full].mean()
        contrast = abs(I_in - I_out) / rng
        if contrast >= contrast_thr:
            restore[rr, cc] = True

    m_final = m_filled & (~restore)
    m_final = remove_small_objects(m_final, min_size=min_fg)
    return m_final

# ==============================
# Patch-based prediction
# ==============================
def predict_full_image_prob(model, image, patch_size=256):
    # If you prefer to infer from the model instead of hardcoding:
    # h_in, w_in = model.input_shape[1:3]; patch_size = h_in or patch_size

    H, W = image.shape[:2]
    prob_map  = np.zeros((H, W), dtype=np.float32)
    count_map = np.zeros((H, W), dtype=np.float32)

    for i in range(0, H, patch_size):
        for j in range(0, W, patch_size):
            # take whatever fits at the border
            patch = image[i:i+patch_size, j:j+patch_size]
            ph, pw = patch.shape[:2]

            # normalize to float32 [0,1], add channel dim
            pnorm = normalize(np.asarray(patch, dtype=np.float32), axis=1)[..., np.newaxis]

            # pad to (patch_size, patch_size, 1) for the model
            if ph < patch_size or pw < patch_size:
                canvas = np.zeros((patch_size, patch_size, 1), dtype=np.float32)
                canvas[:ph, :pw, 0] = pnorm[..., 0]
            else:
                canvas = pnorm

            pin = canvas[np.newaxis, ...]  # (1, H, W, 1)

            # predict -> (1, patch_size, patch_size, 1)
            out = model.predict(pin, verbose=0)
            if isinstance(out, (list, tuple)): out = out[0]
            pred_full = np.asarray(out, dtype=np.float32).squeeze()
            if pred_full.ndim == 3: pred_full = pred_full[..., 0]  # (patch_size, patch_size)

            # crop back to the original patch region (ph, pw)
            pred = pred_full[:ph, :pw]

            # write into full canvas
            prob_map[i:i+ph, j:j+pw]  += pred
            count_map[i:i+ph, j:j+pw] += 1.0

    count_map[count_map == 0] = 1.0
    prob_map /= count_map
    return np.clip(prob_map, 0, 1).astype(np.float32)



# ==============================
# Per-image, multi-model panel
# ==============================
def visualize_models_on_image(models, img, thr=0.6, use_post=True, pp_kwargs=None,
                              show_probs=False, gt=None, savepath=None, title=""):
    """
    models: dict[name->keras.Model]
    img   : (H,W) grayscale
    gt    : optional ground-truth mask (H,W) in {0,1}
    """
    pp_kwargs = pp_kwargs or {}
    preds = []

    # get display base
    img_disp = np.asarray(img, dtype=np.float32)

    # compute per-model prob & mask
    for name, m in models.items():
        P = predict_full_image_prob(m, img_disp, patch_size=256)
        if use_post:
            mask = clean_mask_with_contrast(P, img_disp, thr=thr, **pp_kwargs)
        else:
            mask = P >= thr

        # metrics if GT is provided
        iou = dice = None
        if gt is not None:
            gt_b = gt > 0.5
            inter = int(np.logical_and(gt_b, mask).sum())
            union = int(np.logical_or(gt_b, mask).sum())
            iou = (inter / union) if union > 0 else 1.0
            denom = int(gt_b.sum()) + int(mask.sum())
            dice = (2.0 * inter / denom) if denom > 0 else 1.0

        preds.append((name, P, mask, iou, dice))

    # layout: image, (optional GT), then each model
    extra = 1 + (1 if gt is not None else 0)
    panels = extra + len(preds)
    cols = min(panels, 6)
    rows = math.ceil(panels / cols)
    fig = plt.figure(figsize=(4 * cols, 4 * rows))

    # 1) images image
    ax = plt.subplot(rows, cols, 1)
    ax.set_title(f"{title} raw")
    ax.imshow(img_disp, cmap="gray"); ax.axis("off")

    # 2) GT (optional)
    k = 2
    if gt is not None:
        ax = plt.subplot(rows, cols, k); k += 1
        ax.set_title("GT")
        ax.imshow(img_disp, cmap="gray")
        ax.imshow(gt > 0.5, alpha=0.4); ax.axis("off")

    # 3..) each model
    for name, P, mask, iou, dice in preds:
        ax = plt.subplot(rows, cols, k); k += 1
        tag = " +pp" if use_post else ""
        met = ""
        if iou is not None:
            met = f"\nIoU={iou:.3f} Dice={dice:.3f}"
        ax.set_title(f"{name}{tag}{met}")
        ax.imshow(img_disp, cmap="gray")
        overlay = P if show_probs else mask
        im = ax.imshow(overlay, alpha=0.4)
        ax.axis("off")
        if show_probs:
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    if savepath:
        plt.savefig(savepath, dpi=160, bbox_inches="tight")
        plt.show()
        plt.close(fig)
    else:
        plt.show()

# ==============================
# Directory runner
# ==============================
def run_dir_multimodel(models, img_dir, out_dir,
                       thr=0.6, use_post=True, pp_kwargs=None,
                       patterns=("*.png","*.jpg","*.jpeg","*.tif","*.tiff"),
                       show_probs=False, gt_suffix=None):
    """
    gt_suffix: if you have GT masks, set e.g. '_mask.png' to load file.stem+gt_suffix next to the image
    """
    img_dir = Path(img_dir); out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    files = []
    for pat in patterns:
        files.extend(sorted(img_dir.glob(pat)))

    for f in files:
        print(f"\n=== {f.name} ===")
        img = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
        if img is None:
            print("  (skipped: unreadable)"); continue

        gt = None
        if gt_suffix:
            gt_path = f.with_name(f.stem + gt_suffix)
            if gt_path.exists():
                gt = cv2.imread(str(gt_path), cv2.IMREAD_GRAYSCALE)
                if gt is not None: gt = (gt > 127).astype(np.uint8)

        savepath = out_dir / f"{f.stem}_panel.png"
        visualize_models_on_image(
            models, img, thr=thr, use_post=use_post, pp_kwargs=pp_kwargs or {},
            show_probs=show_probs, gt=gt, savepath=str(savepath), title=f.stem
        )
        print(f"Saved {savepath}")


In [ ]:

# --- Example usage ---
# from tensorflow.keras.models import load_model
# model = load_model("path/to/your_trained_model.h5",
#                    custom_objects={"dice_coef": dice_coef, ...})  # add custom losses/metrics if needed
#
# run_on_directory(
#     model,
#     "/home/anvy4548/projects/crystal-recognition/test_images",
#     "/home/anvy4548/projects/crystal-recognition/results",
#     patch_size=256,
#     thr=0.5
# )

In [ ]:
img_root = Path("/home/anvy4548/projects/crystal-recognition/test_images")
out_root = Path("outputs/panels"); out_root.mkdir(parents=True, exist_ok=True)

pp = dict(hole_area=600, min_blob=400, max_blob=6000,
          prob_bg=0.3, contrast_thr=0.10, ring_r=3, close_r=3, min_fg=150)

run_dir_multimodel(
    models,
    img_dir=img_root,
    out_dir=out_root,
    thr=0.6,
    use_post=True,
    pp_kwargs=pp,
    show_probs=False,
    gt_suffix=None
)


In [ ]:
img_root = Path("/home/anvy4548/projects/crystal-recognition/test_images")
out_root = Path("outputs/panels"); out_root.mkdir(parents=True, exist_ok=True)

pp = dict(hole_area=600, min_blob=400, max_blob=6000,
          prob_bg=0.3, contrast_thr=0.10, ring_r=3, close_r=3, min_fg=150)

run_dir_multimodel(
    models,
    img_dir=img_root,
    out_dir=out_root,
    thr=0.6,
    use_post=False,
    show_probs=False,
    gt_suffix=None
)

In [ ]:
img_root = Path("/home/anvy4548/projects/crystal-recognition/test_images")
out_root = Path("outputs/panels-pp");
out_root.mkdir(parents=True, exist_ok=True)

pp = dict(hole_area=600, min_blob=400, max_blob=6000,
          prob_bg=0.3, contrast_thr=0.10, ring_r=3, close_r=3, min_fg=150)

run_dir_multimodel(
    models,
    img_dir=img_root,
    out_dir=out_root,
    thr=0.6,
    use_post=True,
    pp_kwargs=pp,
    show_probs=False,
    gt_suffix=None
)